In [ ]:
import pandas as pd
import numpy as np
import os

def generate_concentration_data(market_names, events_dir, output_dir=None):
    """
    For each market, compute per‑timestamp concentration of supply (collateral) and borrow (debt).

    Parameters:
    market_names : list of market names (file names without .csv)
    events_dir  : directory containing enriched event CSV files
    output_dir  : optional directory to save CSV files

    Returns:
    df_supply, df_borrow : DataFrames with columns:
        market, timestamp, datetime, side, user_address, share (percentage),
        total (total supply or borrow at that timestamp),
        hhi (Herfindahl‑Hirschman Index), n_active (number of active users)
    """
    supply_records = []
    borrow_records = []

    for market in market_names:
        events_path = os.path.join(events_dir, f"{market}.csv")
        if not os.path.exists(events_path):
            print(f"File not found: {events_path}")
            continue

        df = pd.read_csv(events_path)
        if 'datetime' not in df.columns and 'timestamp' in df.columns:
            df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
        df = df.sort_values(['timestamp', 'hash']).reset_index(drop=True)

        # State dictionaries: user -> amount
        user_supply = {}   # collateral_after
        user_debt = {}     # debt_after

        # Group events by exact timestamp (second precision)
        grouped = df.groupby('timestamp')
        for ts, group in grouped:
            # Apply all events in this timestamp in order (already sorted by hash within group)
            for _, row in group.iterrows():
                user = row['user_address']
                # Update supply (collateral)
                if 'collateral_after' in row and pd.notna(row['collateral_after']):
                    user_supply[user] = row['collateral_after']
                # Update debt
                if 'debt_after' in row and pd.notna(row['debt_after']):
                    user_debt[user] = row['debt_after']
                # Note: we do not delete users when amount becomes zero; they will have zero contribution.
            # After processing all events at this timestamp, compute metrics
            datetime_val = group['datetime'].iloc[0]

            # ---- Supply (collateral) ----
            total_supply = sum(user_supply.values())
            if total_supply > 0:
                shares = {u: v / total_supply * 100 for u, v in user_supply.items() if v > 0}
                large = {u: s for u, s in shares.items() if s > 5}
                other_share = sum(s for u, s in shares.items() if u not in large)
                hhi = sum((s / 100) ** 2 for s in shares.values())
                n_active = len([u for u, v in user_supply.items() if v > 0])

                for u, s in large.items():
                    supply_records.append({
                        'market': market,
                        'timestamp': ts,
                        'datetime': datetime_val,
                        'side': 'supply',
                        'user_address': u,
                        'share': s,
                        'total': total_supply,
                        'hhi': hhi,
                        'n_active': n_active
                    })
                if other_share > 0:
                    supply_records.append({
                        'market': market,
                        'timestamp': ts,
                        'datetime': datetime_val,
                        'side': 'supply',
                        'user_address': 'other',
                        'share': other_share,
                        'total': total_supply,
                        'hhi': hhi,
                        'n_active': n_active
                    })

            # ---- Borrow (debt) ----
            total_borrow = sum(user_debt.values())
            if total_borrow > 0:
                shares = {u: v / total_borrow * 100 for u, v in user_debt.items() if v > 0}
                large = {u: s for u, s in shares.items() if s > 5}
                other_share = sum(s for u, s in shares.items() if u not in large)
                hhi = sum((s / 100) ** 2 for s in shares.values())
                n_active = len([u for u, v in user_debt.items() if v > 0])

                for u, s in large.items():
                    borrow_records.append({
                        'market': market,
                        'timestamp': ts,
                        'datetime': datetime_val,
                        'side': 'borrow',
                        'user_address': u,
                        'share': s,
                        'total': total_borrow,
                        'hhi': hhi,
                        'n_active': n_active
                    })
                if other_share > 0:
                    borrow_records.append({
                        'market': market,
                        'timestamp': ts,
                        'datetime': datetime_val,
                        'side': 'borrow',
                        'user_address': 'other',
                        'share': other_share,
                        'total': total_borrow,
                        'hhi': hhi,
                        'n_active': n_active
                    })

    df_supply = pd.DataFrame(supply_records)
    df_borrow = pd.DataFrame(borrow_records)

    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        df_supply.to_csv(os.path.join(output_dir, 'supply_concentration.csv'), index=False)
        df_borrow.to_csv(os.path.join(output_dir, 'borrow_concentration.csv'), index=False)
        print(f"Saved supply and borrow concentration data to {output_dir}")

    return df_supply, df_borrow


In [9]:
df_supply, df_borrow = generate_concentration_data(
    [
        "eth_syrupusdc_usdc",
        "eth_usr_usdc",
        "eth_rlp_usdc",
        "eth_wstusr_usdc",
        "eth_fxsave_usdc",
        "eth_siusd_usdc",
        "eth_reusd_usdc",
        "eth_siusd_usdc",
        "eth_stcusd_usdc",
    ],
    "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_enriched",
)

/var/folders/hj/pbs977kd43s6n1l9z3mxrj200000gn/T/ipykernel_56842/1222699717.py:29: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(events_path)


In [10]:
df_supply.to_csv("/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_suppliers_share.csv", index=False)
df_borrow.to_csv("/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_borrowers_share.csv", index=False)
